# 📘 2.3 Systems of Linear Equations
### Matrices, Gauss-Jordan, LU Decomposition, Jacobi Method and Singular Value Decomposition

This section explores the numerical solution of systems of linear equations, looking at different methods that drastically impact the required number of FLOPs and memory requirements when you need to solve large matrices. Interative methods are introducted that enable further trades-offs between required accuracy and computational costs. Finally we look at how decompositions can capture the numerical nature of a linear system.

---

### 🎯 Learning Objectives
> By the end of this notebook, you should be able to:
> * Recognise the significance of the numerical solution of linear systems in engineering.
> * Be able to identify a linear system of equations.
> * Understand row and column degeneracy and how they impact the solution of linear systems, and appreciate how numerical precision can render an equation numerically singular.
> * Understand the range of potential numerical solutions to linear systems and the potential benefits and drawbacks to a decomposition or iterative approach.
> * Know how to test for a singular system and solve for over- or under-constrained systems.

## 📂 2.3.1 The Significance of Linear Algebraic Computations

---

Linear alegbra places a key roles in numerous applications, and many partical engineering methods for control and stability are geared around developing linear representations of system, as it is often only these linear equations that we have a chance of solving.

The Graphical Processing Unit (GPU) in your computer, and those that form the backbone of modern AI, have specialised tensor cores that are designed and optimised for fast matrix multiplications (used in geometric transformations and neural networks). Most GPU handle simple fused multiply add (AxB+C) operations, and can do Trillions of these per-second. Any problem that can be reduced to a large number of simultaneous AxB+C operations can therefore be run rapidly on a GPU. These are precisely the operations you need for solve large arrays of equations.

The video below give a good idea of the relative computational power you can get from a GPU. You don't need to understand exactly how it achieves this, but note that they excel at the *embarrassingly parallel* calculations and repeated operations that form the basis of linear algebra operations.


> ### 📖 Definition
> **embarrassingly parallel** computations are those that can be split into completely independent tasks with no need for communication or coordination between them.
>
> For example, the multiplication of a large matrix requires repeatative multiplication and of each element in each row with each elements in each column. The product of each element and the addition of these can be done independent of the rest of the values, making it easy to divide the problem up into independent operations. You could say that it is *embarrassingly* easy to *parallelise*.

In [ ]:
# @title What does a GPU have to do with Linear Algebra?
from IPython.display import HTML
HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/h9Z4oGN89MU?si=kpon6O2WCUPvh844" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen referrerpolicy="strict-origin-when-cross-origin"></iframe>')

The video above focuses on some of the mathematical operations required in 3D rendering and computer games, but these are also the methods and algorithms used in medical CAT scans, machine visions and robotics.

> ### 💬 Aside
> For example the rotation of a coordinate system about the x,y,z axes can be applied by multiplication of point coordinates with the following matrices, respectively.
>
>\begin{equation} {\bf R}_x=
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos\theta & -\sin\theta \\
0 & \sin\theta & \cos\theta
\end{bmatrix} \tag{2.26a}
\end{equation}
>
>\begin{equation} {\bf R}_y=
\begin{bmatrix}
\cos\theta & 0 & \sin\theta \\
0 & 1 & 0 \\
-\sin\theta & 0 & \cos\theta
\end{bmatrix}
\tag{2.26b}
\end{equation}
>
>\begin{equation} {\bf R}_z=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{bmatrix}\tag{2.26c}
\end{equation}
>
>where the rotation of point ${\bf P} = [x\mbox{ }y\mbox{ }z]^T$ around the $x$-axis is given by
>
>\begin{equation}
{\bf P}' = {\bf R}_x {\bf P}.
\end{equation}
>
>This would have to be performed for each point in the domain.

### 🔹 Linear vs Non-Linear Algebraic Equations


Before we proceed, we had better be sure that we all understand what we mean by a linear equation and how to recognise one.

> ### 📖 Definition
> **Linear Algebraic Equations** are ones where each unknown (e.g. ${\bf x}$) are related to constants $\bf b$, through series of constants coefficients $a_{ij}$, i.e.
>
>\begin{aligned}
a_{11}x_1 + a_{12}x_2 + a_{13}x_3 &= b_1 \\
a_{21}x_1 + a_{22}x_2 + a_{23}x_3 &= b_2 \\
a_{31}x_1 + a_{32}x_2 + a_{33}x_3 &= b_3
\end{aligned}
>
> **Linearity** in mathematics describes a relationship where changes in input produce proportional changes in output, adhering to the principles of
>* additivity $f(x+y)=f(x)+f(y)$ and
>* homogeneity $f(ax)=af(x)$.

The function of a straight line, $y=mx+c$, is an simple example of a linear equation, if $m$ and $c$ are constants and independent of $x$. If we double $x$ we will see a proporsional increase in y.

The rotation matrices in equations 2.26a-c are also linear for any constant $\theta$.

So what would a non-linear equation look like?

> ### 📖 Definition
> **non-linear equation** is an equation in which at least one variable, $x_i$:
> * appears with an exponent other than 1 (e.g. $x^2$)
> * is multiplied by another variable (e.g. $x_1\cdot x_2$)
> * is placed inside a nonlinear function (such as a square root, trigonometric function, exponential, or logarithm).

For example, $f(x) =ax^2 + b$ is non-linear. We can see it is non-additive, i.e. we can split the input, e.g.\
\begin{equation}
f(x+y) = a(x+y)^2+b \ne ax^2+b + ay^2+b \ne f(x) + f(y),
\end{equation}
and non-homogeneous
$$ f(cx) = a(cx)^2 + b \ne c(ax^2+b) \ne cf(x).$$

> ### ⚠️ Point of note
> A **non-linear** equation **cannot** be written as a linear combination of coefficents and variables.

It is this distinction, which makes *linear* equations, easy to solve and some *non-linear* equation impossible - *see the Navier-Stokes equations for example.*

As Linear equations only depend on products of variables with constant, they can be represented in matrix notation as
\begin{equation}
\bf A\cdot x = b
\end{equation}
where the raised dot here indicates matrix muplitication. A problem of $N$ *unknowns* and $M$ *equations* can be expanded in matrix form as

\begin{equation}
\mathbf{A} = \begin{bmatrix}
a_{11} & a_{12} & \cdots & a_{1N} \\
a_{21} & a_{22} & \cdots & a_{2N} \\
\vdots & \vdots & \ddots & \vdots \\
a_{M1} & a_{M2} & \cdots & a_{MN}
\end{bmatrix}
\end{equation}

and

\begin{equation}
\mathbf{b} = \begin{bmatrix}
b_{1} \\
b_{2} \\
\vdots \\
b_{M}
\end{bmatrix}
\end{equation}

## 📂 2.3.2 Non-Singular verses Singular Sets of Equations

---

When we think about linear equations, we often think about nice neat problems where we have $N$ unknowns and an equal number, $M=N$, equations. In such cases we have a good chance of solving for unique set of values $x_j$, where ${\bf x} = [x_1\mbox{ }x_2\dots x_N]^T$ and $j = 1$ to $N$. I say a good chance, as this is not guaranteed.

A unique solution will **NOT** exist if
* one or more of the equations (i.e. rows of matrix $\bf A$) is a linear combination of the others - termed *row degeneracy*.
>e.g. 3x4 row-degenerate matrix, where the third row is a linear combination of the first two, could look like this:
>
>$$\mathbf{A} = \begin{bmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
6 & 8 & 10 & 12
\end{bmatrix}$$
>
>Upon a cursory glance this seems fine. However, the third row is the sum of the first two rows ($[1+5, 2+6, 3+7, 4+8] = [6, 8, 10, 12]$). Which means these equations aren't independent.

* all equations contain one or more variables in the same linear combination - termed *column degeneracy*.
> e.g.
>$$\mathbf{A} = \begin{bmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 11 & 8 \\
9 & 10 & 19 & 12
\end{bmatrix}$$
>
>Here, the third column is the sum of the first and second columns ($[1+2, 5+6, 9+10] = [3, 11, 19]$).


Such set of equations results in $\bf A$ being termed a **singular** matrix.

>### 📖 Definition
> A **Singular Matrix:**
> * has a zero determinant, $\det({\bf A}) = 0$
> * has linearly dependent rows or columns
> * has a rank is less than its dimension (e.g., a $3\times 3$ matrix with rank 1 or 2)
> * the inverse, i.e., $\mathbf{A}^{-1}$, does not exist
> * results in the equation $(\mathbf{A}\cdot \mathbf{x} = \mathbf{b})$ not have a unique solution

If **N=M** then *row degeneracy* implies *column degeneracy* and vice-versa.

> ### 💬 Aside
> You can prompt AI to produce examples of singular matrices if you'd like to see more examples. It's a strange New-World we inhabit!

### 🧪 Example of Implication of Degenerate System of Linear Equations

Consider a 3x3 system of linear equations which consists of $N=3$ unknowns and $M=3$ equations

$$\begin{aligned}
x + 2y + 3z &= 6 \\
4x + 5y + 6z &= 15 \\
5x + 7y + 9z &= 21
\end{aligned}$$

or in matrix form

$$\begin{bmatrix}
1 & 2 & 3\\
4 & 5 & 6\\
5 & 7 & 9
\end{bmatrix}
\begin{bmatrix}
x\\ y\\ z
\end{bmatrix}
= \begin{bmatrix}
6\\ 15\\ 21
\end{bmatrix}$$

In this system, the third equation ($5x + 7y + 9z = 21$) is the sum of the first two equations,
$$(x + 2y + 3z) + (4x + 5y + 6z) = 5x + 7y + 9z,$$
 and the sum of their right-hand sides ($6 + 15 = 21$). This system exhibits **row degeneracy**.

As this is a square matrix, the fact that it is row degenerate, means it is also **column degenerate** as column 3 $= -1\times$ column 1 $+ 2 \times$ column 2. (I asked AI to show me this). Finding this degeneracy is less straightforward, but there is still a linear combination of column, hence degeneracy.

What about it's determinant?
$$ \det(\mathbf{A}) = 1\left(5\times 9 - 6\times 7\right) -2\left(4\times 9 - 5\times 6\right)+3\left(4\times 7 - 5\times 5\right) = 3 -12 +9 = 0$$

The determinant is a separate way we can check if the matrix is singular.

What are the implications of this on our ability to solve for the variables, $x,y,z$. Let's think about how you would have solved this in high school. You'd probably try to cancel a variable, for instance let's try to get ride of $x$ by considering a linear combination of the equations.

E.g. equation (3) - (2) =
$$x+2y+3z = 6 \tag{4'}$$

Then we can cancel $x$ from equation (1) by (1) - (4')

Unfortunately, (4') is the same as equation (1). If we now subtract this we not only cancel $x$, but we also cancel $y,z$. This is the direct consequence of the degeneracy of this matrix.

> ### ⚠️ Point of note
> There is no **unique solution** for this problem - but there are still *possible* solutions. Sometime it is only a possible solution we are after. Some solutions that are mathematically possible might no be physical. A complex number is a good example of this.



## 📂 2.3.3 Impact of Numerical Precision on Systems of Linear Equations

---

As we have seen from our consideration of numerical errors ([Notebook 2.2](https://colab.research.google.com/drive/1MSTMa20hP4VybIFfmPwbaUGAtWGEDUxC?usp=playground)), rounding errors and numerical precision can leads to cases where mathematically distinct numbers become indistinguishable to the computer (difference less than machine precision).

In such cases, matrices and systems of equations that are not strictly degenerate can become numerically indistinguishable from those that are and may cause solution methods to fail in the same way. In such cases we would say that we a **Numerical Singular Matrix**.

> ### 📖 Definition
>A **numerically singular matrix** is a matrix that is not exactly *singular* in theory, but behaves as if it were singular when you perform computations on a computer. In other words, its determinant isn't exactly zero, but it is so close to zero that floating-point arithmetic cannot reliably distinguish it from a truly singular matrix.

Large systems of equations are represented by large matrices, which are consequently suseptible to the accumulation of **rounding errors**. This may not lead to the failure of an algorith, but may return a series of calculated variables $\bf x$, that don't satisfy the original equations when you substitute them back in.

Numerous algorithms have therefore been developed to solve linear algebra problems, with tradeoffs in computational time, cost and accuracy.

Unfortunately, it is not always easy to tell when either of these problems will occur.
>Press, W. H., Teukolsky, S. A., Vetterling, W. T., & Flannery, B. P. (2002). Numerical Recipes in C++: The Art of Scientific Computing (2nd ed.).
>
> Give the following guidelines:
>* Linear equation with variable number $N < 20$ to 50 can generally be solved in 32-Bit single precision without using special methods (as long as equations aren't singular).
>* Linear equations with upto $ N \approx 1000$ variable can generally be solved in 64-Bit double precision with limits being more computational time rather than accuracy.
>* Sets of $N=1000$'s or more can still be solved if matrices are sparse (i.e. many coefficients are zero)
>* Even a small problem of $N =10$ may need special methods if it is close to *singular* $\rightarrow$ *numerically singular*.

**What does this mean in practice?**

Asking AI how to solve a matrix might note be enough to guide you to an effective soluition as the solution proposed might not be well-suited to your problem! The optimal method for a small test case might also fail when you apply it much larger datasets.

I guess that's why we have a whole notebook on Linear Algebra.

## 📂 2.3.4 Direct Methods of Solving $\bf Ax = b$

---

From earlier mathematics units, you should be familiar with the basic principle of solving a matrix by inversion, i.e.

* calculate the inverse of matrix $\bf A^{-1}$
* multiply both sides of the equaton by the inverse $\bf A^{-1}$
\begin{align}
\bf A^{-1} A x &=& \bf A^{-1}b\\
\bf I x &=& \bf A^{-1}b\\
\bf x &=& \bf A^{-1}b
\end{align}
where $\bf I$ is the identity matrix.

This approach quickly becomes prohibitively expensive as it requires the full inversion.

Let's consider/reconsider some alternatives.

### 🔹 2.3.4.1 Gauss-Jordan Elimination



In ENG1014 you covered *Gaussian Elimination* (or *row reduction) and probably the related Gaussian-Jordan Elimination*, which is a systematic way of calculating the inverse of a matrix.

*So this should be revision!*

Recall that Gaussian Elimination involves the following process, given a matrix $\bf A$,

\begin{equation}
\bf A = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1N}\\
a_{21} & a_{22} & \cdots & a_{2N}\\
\vdots & \vdots &        & \vdots\\
a_{M1} & a_{M2} & \cdots & a_{MN}
\end{bmatrix}
\end{equation}

1.   Form an *augmented* matrix,$[A_{MN} \mid I_N]$ by placed the *identity* matrix, $\bf I$ on the right hand side of matrix $\bf A$

\begin{equation}
\left[
\begin{array}{cccc|cccc}
a_{11} & a_{12} & \cdots & a_{1N} & 1 & 0 & \cdots & 0 \\
a_{21} & a_{22} & \cdots & a_{2N} & 0 & 1 & \cdots & 0 \\
\vdots & \vdots &        & \vdots & \vdots & \vdots & \ddots & \vdots \\
a_{M1} & a_{M2} & \cdots & a_{MN} & 0 & 0 & \cdots & 1
\end{array}
\right]
\end{equation}

2.   Perform *Forward elimination* to **upper triangular / row-echelon form**
* use a sequence of elementary row operations (i.e. swapping rows, multiplying a row by a non-zero number and adding a multiple of one row to another) to eliminate non-zero values below the diagagonal, until the augmented matrix is of the form
$$\left[
\begin{array}{cccc|cccc}
* & * & \cdots & * & * & * & \cdots & * \\
0 & * & \cdots & * & * & * & \cdots & * \\
\vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & * & * & * & \cdots & *
\end{array}
\right]$$
where * can be zero or non-zero values, but lower left values must be zero. This process is termed **Gaussian Elimination**.

3. In **Gaussian-Jordan Elimination**, you continue row operations to remove non-zero terms from above the diagonal of the lefthand side of the matrix to arrive at the *reduced row-echelon form*.
$$\left[
\begin{array}{cccc|cccc}
1 & 0 & \cdots & 0 & c_{11} & c_{12} & \cdots & c_{1n} \\
0 & 1 & \cdots & 0 & c_{21} & c_{22} & \cdots & c_{2n} \\
\vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & 1 & c_{n1} & c_{n2} & \cdots & c_{nn}
\end{array}\right]
$$
where the right-hand side of the augmented matrix is now the inverse of $\bf A$, i.e. $\bf C = A^{-1}$.  

> If this doesn't ring a bell, don't worry. I will take you through this process in a video at the end of this section.

You should also be familar with the direct use of *Gaussian Elimination* in the solution of the linear equations, where you multiple the identity matrix by the matrix $\bf B$, such that the augmented matrix is now
\begin{equation}
\left[
\begin{array}{cccc|c}
a_{11} & a_{12} & \cdots & a_{1N} & b_1 \\
a_{21} & a_{22} & \cdots & a_{2N} & b_2 \\
\vdots & \vdots &        & \vdots & \vdots \\
a_{M1} & a_{M2} & \cdots & a_{MN} & b_M
\end{array}
\right]
\end{equation}
Row operations are again used to get to the *upper triangular* or *row echelon form*,
$$\left[
\begin{array}{cccc|c}
* & * & \cdots & * & * \\
0 & * & \cdots & * & * \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & \cdots & * & *
\end{array}
\right]$$
which equates to
$$\left[
\begin{array}{cccc}
d_{11} & d_{12} & \cdots & d_{1N} \\
0 & d_{22} & \cdots & d_{2N}  \\
\vdots & \vdots & \ddots & \vdots  \\
0 & 0 & \cdots & d_{MN}
\end{array}
\right]
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots\\
x_N
\end{bmatrix}
=
\begin{bmatrix}
e_1 \\
e_2 \\
\vdots\\
e_M
\end{bmatrix}
$$
from which $x_N = e_M/d_{MN}$ which can then be back-substituted into the rows above the diagonal to solve for $x_{N-1}$ and so on.

Alternatively *Gauss-Jordan Elimination* yields
$$\left[
\begin{array}{cccc}
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0  \\
\vdots & \vdots & \ddots & \vdots  \\
0 & 0 & \cdots & 1
\end{array}
\right]
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots\\
x_N
\end{bmatrix}
=
\begin{bmatrix}
f_1 \\
f_2 \\
\vdots\\
f_M
\end{bmatrix}
$$
after row operations, such that the values of $\bf x$ equate to those of the final column of the augmented matrix $\bf F$.

**If you need to seem some examples of this process, ask AI. It will be happy to generate endless examples**.


A Mermaid flow chart for this - as generate by the prompt

>### 🗣️ AI Prompts
> In this case feed the prompt 'generate a mermaid flow chart for gauss-jordan elimination to solve Ax=B' to MS Copilot - SMART (June 26).

Is presented below. I had to manually modify this to remove some non-standard ASCII characters like $\ne$ and $^{-1}$ to get this to render without throwing a syntex error. The current *Pivot* column in the figures refers to the column which you are currently trying to set all values below to zero as you go through the process.

In [ ]:
# @title Flow Chart Gauss-Jordan Eliminaton
# Try to access the colab_print tool
try: from colab_print import mermaid
except:
  # access failed to install it and try again
  !pip install colab-print
  from colab_print import mermaid

mermaid('''
flowchart TD;
A0(["Start with augmented matrix [A | B]"]) --> A1
A1["Select pivot column k = 1"] --> A2
A2{"Is pivot a_kk != 0?"} -->|Yes| A4
A2 -->|No| A3
A3["Swap with a lower row having nonzero entry"] --> A4
A4["Normalize pivot row: Rk <-- Rk / a_kk"] --> A5
A5["Eliminate entries below pivot: Ri <-- Ri - a_ik Rk"] --> A6
A6{"More columns left?"} -->|Yes| A7
A6 -->|No| B0
A7[Increment k --> next pivot column] --> A1
B0[Begin backward elimination] --> B1
B1[For each pivot row k from n down to 1] --> B2
B2["Eliminate entries above pivot: Ri <-- Ri - a_ik Rk"] --> B3
B3{More pivot rows?} -->|Yes| B1
B3 -->|No| C0
C0[Left block becomes identity matrix I] --> C1
C1[Right block is the solution X = A^-1 B] --> END
END([End])
''')


> ### ⚠️ Point of note
> I have experienced some odd clipping of text in Mermaid flow charts. It seems to be associated with changes to the browser zoom. You may need to reset your zoom and re-run these is this happends to you.

The following videos walk you through an example of the manual process and steps required to:
* get from the original matrix to its row echelon form - via **Gaussian Elimination**;
* use the process of **back substituion** to solve a linear system directly from its row echelon form;
* peform a **backward elimination** to complete Gauss-Jordan process and get to a reduce row echelon form.

In [ ]:
# @title  🎥 Video Walkthrough of Gaussian Elimination Using an Example Non-Singular Matrix Generated by AI. (The video is not AI 😜) (5:54)
from IPython.display import IFrame
IFrame(src="https://drive.google.com/file/d/1O6YTLYsU3qzt181ss3zfGeABuxc2bgeG/preview", width="640", height="360")

In [ ]:
# @title  🎥 Video Walkthrough of Back Substitution to Solve a Row Echelon System (15:37)
from IPython.display import IFrame
IFrame(src="https://drive.google.com/file/d/1mdzkVNh9rbmU5kjCeahDU99d6_j4_o-I/preview", width="640", height="360")

In [ ]:
# @title  🎥 Video Walkthrough of Backward Elimination to Complete Gauss-Jordan Solution (8:33)
from IPython.display import IFrame
IFrame(src="https://drive.google.com/file/d/1VvT4kOQFOm4EAZ3F0tz8Y6PnqSs_sltX/preview", width="640", height="360")

### 🔹 2.3.4.2 Operations Required for Gauss-Jordan Elimination

In order to assess an algorithm, we need to consider the computational cost (FLOPs, Memory usage), accuracy and stability.

The exact number of operations required will often depend on the input data, but you can generally get an idea of the order of the operations that are required.

In this section, I will walk you through the method, and generate a list of steps involved in the process and the number of operations as functions of the matrix size.

Consider Gauss-Jordan Elimination of an $M\times N$ square matrix ($N=M$), where $M$ is the number of equations and $N$ is the number of variables in a linear system of equations.
\begin{equation}
\left[
\begin{array}{cccc|c}
a_{1,j=1} & a_{12} & \cdots & a_{1N} & b_1 \\
a_{2,j=1} & a_{22} & \cdots & a_{2N} & b_2 \\
\vdots & \vdots &        & \vdots & \vdots \\
a_{M,j=1} & a_{M2} & \cdots & a_{MN} & b_M
\end{array}
\right]
\end{equation}

Starting with the **forward elimination** to the *upper diagonal/row echelon form*, assuming that none of the elements below the diagonal are initially zero and the diagonal terms are not already equal to 1, consider the first *pivot* point being $a_{kk}$ (You can follow along with the flow chart if it helps). Normalising this to unity (1) requires dividing every value along that row $i=k$ (i.e. for $k =1$, $a_{11}, \dots , a_{1N}, b_1$) by $a_{kk}$. As we move to higher values of $k$, and the terms below the pivot point have been set to zero, the number of non-zero terms in the row is reduced. If $k$ is the current row in which the *pivot* is placed, then

> 1. rescaling row $k$ requires $N-(k-1)= N+1-k$ division operations.

**note**: we are presently considering operations on the left-hand side of the augmented matrix ($\bf{A}$), only.
Not the operations on vector $\bf{b}$.

Once rescaled, the algorithm now needs to set values below this pivot to zero, via the multiplication and subtraction of rows.

For any pivot point row $k$, there are now $M-k$ rows of $N+1-k$ columns that need to be set to zero. In each row, each column requires **one** multiplication and **one** subtraction, hence for
> 2. row reduction below pivot point $k$ requires $(M-k)(N+1-k)(2)$ operations

This algorithm requires us to loop through the matrix,  considering each pivot point from $k=1, N$. The total number of operations for pivot rescaling and the forward elimination via steps 1 and 2 becomes

> Operations per Pivot Point $k$
>\begin{eqnarray}
&\sum^{M-1}_{k=1} (M-k)(N+1-k)(2)&+&\sum^{M}_{k=1}(N+1-k),\\
&\text{row reduction} & & \text{row rescaling}
\end{eqnarray}

Given that no reduction is required on the final row, just a rescaling, we can express the total number of operations as

> Total Operations for a $M\times N$ Matrix
>\begin{eqnarray}
&\sum^{M-1}_{k=1}& \left[(M-k)(N+1-k)(2)+(N+1-k)\right]\\
&=& \sum^{M-1}_{k=1}(2M+1)(N+1-k)-2k(N+1-k)\\
&=&\sum^{M-1}_{k=1}\left(2MN+2M-2Mk+N+1-k-2Nk-2k + 2k^2\right)\\
\end{eqnarray}

For square matrices ($N=M$), this reduces to

> Total Operations for a Square Matrix
>\begin{eqnarray}
&\sum^{M-1}_{k=1}& (M-k)(N+1-k)(2)+(N+1-k)\\
&=&\sum^{N-1}_{k=1}\left(2N^2+2N-2Nk+N+1-k-2Nk-2k + 2k^2\right)\\
&=&\sum^{N-1}_{k=1}\left(1+2N^2+3N-4Nk-3k + 2k^2\right)\\
&=&\left( 1+2N^2+3N \right)\sum^{N-1}_{k=1}(1)+2\sum^{N-1}_{k=1}\left(k^2\right) - (4N+3)\sum^{N-1}_{k=1}k \tag{2.27}
\end{eqnarray}

Evaluating these series terms
\begin{eqnarray}
\sum^{N-1}_{k=1}1 &=& N-1\\
\sum_{k=1}^{N-1} k &=& \frac{(N-1)N}{2}\\
&=&\frac{1}{2}N^2-\frac{1}{2}N\\
\sum_{k=1}^{N-1} k^2 &=& \frac{(N-1)N(2N-1)}{6}\\
&=&\frac{1}{3}N^3 - \frac{1}{2}N^2 + \frac{1}{6}N
\end{eqnarray}

Directly substituting these evaluated series terms from the expressions above into Equation 2.27 gives,

\begin{eqnarray}
\left( 1+2N^2+3N \right)(N-1)+2\left(\frac{1}{3}N^3 - \frac{1}{2}N^2 + \frac{1}{6}N\right) - (4N+3)\left(\frac{1}{2}N^2-\frac{1}{2}N \right) \tag{2.27a}\\
\end{eqnarray}

At this point we could expand this, and collect like terms etc., but this isn't really necessary.

A quick glance at the this equation shows that each of there terms has a highest term of the order of $\mathcal{O}(N^3)$ which will be the dominate term. Hence focusing on the highest order term we can reduce equation 2.27a to,
\begin{eqnarray}
\left( 2N^3 + \mathcal{O}(N^2) \right)+\left(\frac{2}{3}N^3 +\mathcal{O}(N^2) \right) - \left(2N^3+\mathcal{O}(N^2)\right) \\
=\frac{2}{3}N^3 + \mathcal{O}(N^2), \tag{2.27b}
\end{eqnarray}

This tells us that the operations, FLOPs, required to take the matrix $\bf{A}$ to its *upper triangular form* are approximately $\frac{2}{3}N^3$ for a $N\times N$ matrix.

>If we consider the operations on the right-hand side of the augmented matrix, which is one column when solving the matrix, then for each pivot point, we have *one* rescaling operation followed by *one* multipication and division for each element below the pivot point of the right-hand side for a total of
>
>\begin{eqnarray}
&\sum^{M-1}_{k=1}& (M-k)(2)+\sum^{M}_{k=1}1
\end{eqnarray}
which for our square matrix becomes approximately
 \begin{eqnarray}
&(2N+1)&\sum^{N-1}_{k=1}1 -2\sum^{N-1}_{k=1}k\\
&=&(2N+1)(N-1)-N^2+N\\
&=&N^2+\mathcal{O}(N)
\end{eqnarray}
Hence, the operation required for the right-hand side are an order of magnitude smaller than that required for the left.

The back-elimination to get the reduce row echelon form in the Gauss-Jordan method, requires the same number of row operations, minus the rescaling - as the diagnonal is already all *ones*. It can be shown that the number of operations require for this back-elimination is $\approx \frac{1}{2}N^3$.

>### ⚠️ Point of note
>*Consider the Implications of this *row reduction* required by both Gaussian Elimination and as the first step of Gauss-Jordan Elimination.*
$$ \text{FLOPS for row reduction} \approx \frac{2}{3}N^3$$
 If the process remains compute-bound at 50 GFLOPs per second, the number of FLOPs and shortest possible compution times are given in Table 1.
>
>**Table 1.** FLOPs and execution time for row reduction of a n×n matrix using the (2/3)n³ FLOP model, assuming sustained performance of 50 GFLOP/s.
>
>|    n     | FLOPs = (2/3)n³        | Time @ 50 GFLOP/s |
>|----------|-------------------------|--------------------|
>|   10     | 6.67e+02               | 1.33e-08 s         |
>|   100    | 6.67e+05               | 1.33e-05 s         |
>|  1000    | 6.67e+08               | 1.33e-02 s         |
>| 10000    | 6.67e+11               | 1.33e+01 s         |
>|100000    | 6.67e+14               | 1.33e+04 s         |
>
>Even at this rate, a problem with 100,000 variables would take over 3.6 hours to compute, neglecting potential accumulation of errors. For 32-Bit values it would also require $6(100,000)^2 = 60 \times 10^9$ Bytes, i.e. 60 GB to store a single matrix.

In [ ]:
# @title  🎥 Video Example of Row Reduction Operations in Gauss-Jordan Elimination (14:17)
from IPython.display import IFrame
IFrame(src="https://drive.google.com/file/d/1iVeIQMw_ClqNKtdF4qsbsiEUPuTQf_Xa/preview", width="640", height="360")

## 📂 2.3.5 LU Decomposition of a Matrix: $\bf L \cdot U = A$

---

Consider what happens if we are able to express our matrix $\bf A$ as the product of both a *lower triangular* and an *upper triangular* matrix, e.g.

\begin{equation} \begin{bmatrix}
\alpha_{11} & 0 & 0 & 0\\
\alpha_{21} & \alpha_{22} & 0 & 0\\
\alpha_{31} & \alpha_{32} & \alpha_{33} & 0\\
\alpha_{41} & \alpha_{42} & \alpha_{43} & \alpha_{44}
\end{bmatrix}
\cdot
\begin{bmatrix}
\beta_{11} & \beta_{12} & \beta_{13} & \beta_{14}\\
0 & \beta_{22} & \beta_{23} & \beta_{24}\\
0 & 0 & \beta_{33} & \beta_{34}\\
0 & 0 & 0 & \beta_{44}
\end{bmatrix}
=
\begin{bmatrix}
a_{11} & a_{12} & a_{13} & a_{14}\\
a_{21} & a_{22} & a_{23} & a_{24}\\
a_{31} & a_{32} & a_{33} & a_{34}\\
a_{41} & a_{42} & a_{43} & a_{44}
\end{bmatrix}.
\end{equation}

Defining the *lower triangular* matrix as $\bf L$ and the *upper triangular* matrix as $\bf U$, we can express this as
\begin{equation}
{\bf L}\cdot{\bf U} = {\bf A} \tag{2.28}
\end{equation}

Consider our original goal of solving a series of linear equations, i.e.
\begin{equation}
{\bf A} \cdot {\bf x} = {\bf b}
\end{equation}

If we subsitute in the triangular forms (i.e. $\mathbf{A}=\mathbf{L}\cdot \mathbf{U}$) this becomes,
\begin{eqnarray}
\left({\bf L} \cdot{\bf U}\right) \cdot {\bf x} &=& {\bf b}\\
{\bf L} \cdot\left({\bf U} \cdot {\bf x}\right) &=& {\bf b}
\end{eqnarray}
since the order of multiplication does not matter here.

If we let $\bf U\cdot x= y$, then we can solve for $\mathbf{x}$ by
1. Solving
$$\mathbf{L}\cdot \mathbf{y} = \mathbf{b}\text{ for }\mathbf{y}, \tag{2.29a}$$
then
2. Solving
$$\mathbf{U}\cdot \mathbf{x} = \mathbf{y}\text{ for }\mathbf{x}. \tag{2.29b}$$

╔═══════════════════════════╗

 ❓ **Why on Earth would we do this?**

 ❓Don't we now need to solve two systems of linear equations.

╚═══════════════════════════╝

We will get there. First we need to consider the FLOPS required to solve a linear system via this method.

### 🔹 2.3.5.1 Forward Substitution to Solve An Lower Triangular System


Think about the cost required to calculate $\mathbf{x}$ via the decomposed form. ${\bf L}$ and ${\bf U}$ have a lot of zero values In solving for $y$ we need to solve

\begin{equation} \begin{bmatrix}
\alpha_{11} & 0 & 0 & 0\\
\alpha_{21} & \alpha_{22} & 0 & 0\\
\alpha_{31} & \alpha_{32} & \alpha_{33} & 0\\
\alpha_{41} & \alpha_{42} & \alpha_{43} & \alpha_{44}
\end{bmatrix}
\cdot
\begin{bmatrix}
y_1\\
y_2\\
y_3\\
y_4
\end{bmatrix}
=
\begin{bmatrix}
b_1\\
b_2\\
b_3\\
b_4
\end{bmatrix},
\end{equation}
followed by
\begin{equation} \begin{bmatrix}
\beta_{11} & \beta_{12} & \beta_{13} & \beta_{14}\\
0 & \beta_{22} & \beta_{23} & \beta_{24}\\
0 & 0 & \beta_{33} & \beta_{34}\\
0 & 0 & 0 & \beta_{44}
\end{bmatrix}
\cdot
\begin{bmatrix}
x_1\\
x_2\\
x_3\\
x_4
\end{bmatrix}
=
\begin{bmatrix}
y_1\\
y_2\\
y_3\\
y_4
\end{bmatrix},
\end{equation}

As these matrix are already in reduced forms, they can be solved using a forward and backwards substitution similar to that required in solution via *Gaussian Elimination*. We skimmed over this before, but let's be more explicit here.

The first row of $\bf L\cdot y = b$ gives us
\begin{eqnarray}
\alpha_{11} y_1 &=& b_1\\
y_1 &=& \frac{b_1}{\alpha_{11}}
\end{eqnarray}
The second row becoming
\begin{eqnarray}
\alpha_{21} y_1 + \alpha_{22}y_2 &=& b_2\\
y_2 &=& \frac{b_2-\alpha_{21}y_1}{\alpha_{22}}
\end{eqnarray}
and so on. As we go down in rows we need to subtract more non-zero terms on the left of the diagonal to solve for $y_i$.

We can express the calculation for each value of $\mathbf{y}$ as
\begin{equation}
y_i = \frac{1}{\alpha_{ii}}\left(b_i-\sum^{i-1}_{j=1} \alpha_{ij}y_j\right)
\end{equation}
which is one division - the term inbrackets divided by $\alpha_{ii}$, $i-1$ multiplications - $\alpha_{ij}y_j$, and $i-1$ subtractions - $-\sum^{i-1}_{j=1}$.

We can express the total number of operations, summed across all, $i$ to $N$, value of $\mathbf{y}$ as,
\begin{eqnarray}
\sum^N_{i=1} \left(1+2(i-1)\right) = 2\sum^N_{i=1}i-\sum^N_{i=1}1
\end{eqnarray}

As we did for the Gauss-Jordan Elimination we can show that, $\sum^N_{i=1} = N$ and $\sum^N_{i=1}i = \frac{1}{2}N(N+1)$, hence
\begin{eqnarray}
2\sum^N_{i=1}i-\sum^N_{i=1}1 = N(N+1)-N = N^2
\end{eqnarray}

>### ⚠️ Point of note
> The solution of $\bf y$ from the *lower triangular* $\bf L$ requires
$$ \text{FLOPS forward substitution} \approx \mathcal{O}(N^2) $$

An additional $\mathcal{O}(N^2)$ operations are required to solve $\bf x$ from the *upper triangular* $\bf U$ and $\bf y$ - which is the mirror of this process.

*Of course, we first need to decompose matrix $\bf A$ in to $\bf L$ and $\bf U$*.



### 🔹 2.2.5.2 Performing the LU Decomposition

How can we extract the matrices $\bf L$ and $\bf U$ from a matrix $\bf A$?

Let's think about the process used in Gaussian Elimination to get to an upper triangular from from a matrix $\bf A$
\begin{equation}
{\bf A} =
\begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}  
\end{bmatrix}.
\end{equation}

While previously we looked as this in terms of operations and the desired outcome, we know need to look at what is mathematically happening to the values in each position within the matrix.

We need to reduce the $j$-th column to zeros below the diagonal, which we can do by row operations for. The $j$-th column will only have non-zero elements below it in rows $i=j+1$ to $N$.

If we define the $i$-th row after these operations as $R'$, then we can express the resulting row in terms of the previous original rows as $R$ as
\begin{equation}
R'_i = R_i-f_{ik}R_{k}
\end{equation}
where $f_{ik} = a_{ik}/a_{kk}$.

E.g. to make the value in the first column $j=1$, below $a_{11}$, equal to zero, we subtract a product of the first row such that the second row becomes
\begin{equation}
R'_2 = R_2 -f_{21}R_{1}
\end{equation}

As we want the value in the place of $a_{21} = 0$, we want $f_{21} = a_{21}/a_{11}$ so that in the $j-th$ column
$$a_{21} - f_{21}a_{11} = 0$$
or
\begin{equation}
R'_2 = R_2 -f_{21}R_{1} = R_2 - \frac{a_{21}}{a_{11}}R_1.
\end{equation}

Similarly if we want the value in the first column ($j=1$) of the third row ($i=4$) to be zero the final row must become
\begin{equation}
R'_3 = R_3 -f_{31}R_{1} = R_3 - \frac{a_{31}}{a_{11}}R_1.
\end{equation}

As these are row operations, they impact the values stored in all columns, hence in reducing the first column, the matrix $\bf A$ has become
\begin{equation}
\bf A' = \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
0 & a_{22}-f_{21}a_{12} & a_{23}-f_{21}a_{13} \\
0 & a_{32}-f_{31}a_{12} & a_{33}-f_{31}a_{13}  
\end{bmatrix}= \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
0 & a'_{22} & a'_{23} \\
0 & a'_{32} & a'_{33}  
\end{bmatrix}.
\end{equation}

*We will use ' to denote the values after the first row reductions that set values in the 1st column to zero and '' the value after setting the second column to zero and so on.*

Reducing the second column, $j=2$, requires

\begin{equation}
R''_3 = R_3' -f_{32}R'_{2},
\end{equation}
where $'$ denotes the modified rows of matrix $\bf A'$. Substituting the modified $a'_{32}$ and $a'_{22}$ values this becomes

\begin{equation}
R''_3 = R_3' -f_{32}R'_{2}=R''_3-\frac{a_{32}-f_{31}a_{12}}{a_{22}-f_{21}a_{12}}R'_2,
\end{equation}
arriving at
\begin{equation}
\bf U= A'' = \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
0 & a_{22}-f_{21}a_{12} & a_{23}-f_{21}a_{13} \\
0 & 0 & a_{33}-f_{31}a_{13}  
\end{bmatrix}
= \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
0 & a'_{22} & a'_{23} \\
0 & 0 & a''_{33}  
\end{bmatrix}
\end{equation}

Above, we now have our *upper triagular* matrix $\bf U$, which when multipled by an as yet unknow *lower triangular* matrix, $\bf L$, must get us back to $\bf A$. i.e.

\begin{equation} \begin{bmatrix}
\alpha_{11} & 0 & 0\\
\alpha_{21} & \alpha_{22} & 0\\
\alpha_{31} & \alpha_{32} & \alpha_{33}
\end{bmatrix}
\cdot
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
0 & a'_{22} & a'_{23} \\
0 & 0 & a''_{33}\\
\end{bmatrix}
=
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
a_{21} & a_{22} & a_{23}\\
a_{31} & a_{32} & a_{33}
\end{bmatrix}.
\end{equation}

If we set the diagonal terms of lower left matrix, $\bf L$, to unity, we can express this as
\begin{equation} \begin{bmatrix}
1 & 0 & 0\\
\alpha_{21} & 1 & 0\\
\alpha_{31} & \alpha_{32} & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
0 & a'_{22} & a'_{23} \\
0 & 0 & a''_{33}\\
\end{bmatrix}
=
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
a_{21} & a_{22} & a_{23}\\
a_{31} & a_{32} & a_{33}
\end{bmatrix},
\end{equation}
the left-hand side of which, after matrix multiplication, is equivalent to

\begin{equation}
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
\alpha_{21}a_{11} & \alpha_{21}a_{12}+a'_{22} & \alpha_{21}a_{13}+a'_{23} \\
\alpha_{31}a_{11} & \alpha_{31}a_{12}+\alpha_{32}a'_{22} & \alpha_{31}a_{13}+\alpha_{32}a'_{23}+a''_{33}\\
\end{bmatrix}
=
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
a_{21} & a_{22} & a_{23}\\
a_{31} & a_{32} & a_{33}
\end{bmatrix},
\end{equation}

Equating terms of the matrix on both sides yields
\begin{eqnarray}
 \alpha_{21}a_{11} &=& a_{21}\\
 \alpha_{21} &=& \frac{a_{21}}{a_{11}}
\end{eqnarray}

Thankfully, this is where something great happens. We allready have a value for $a_{21}/a_{11}$, which we computed during the row reduction required to calculate $\bf U$, i.e.
$$ \alpha_{21} = f_{21}.$$

Infact, we can also show that $\alpha_{31} = f_{31}$ and $\alpha_{32} = f_{32}$, such that the LU decompositon of $\bf A$ can be expressed as

\begin{equation} \begin{bmatrix}
1 & 0 & 0\\
f_{21} & 1 & 0\\
f_{31} & f_{32} & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
0 & a'_{22} & a'_{23} \\
0 & 0 & a''_{33}\\
\end{bmatrix}
=
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
a_{21} & a_{22} & a_{23}\\
a_{31} & a_{32} & a_{33}
\end{bmatrix}.
\end{equation}


>### ⚠️ Point of note
> * The values of $f_{ij}$, $a'_{ij}$ and $a''_{ij}$ need to form the $\mathbf{L}$ and $\mathbf{U}$ can all be computed from the Gaussian Elimination of $\bf A$.
>
> **Importantly** we arrive at $\bf L$ and $\bf U$, *independently* of the right-hand side of the linear system $\bf b$.
> * i.e. LU decomposition is only a function of the coefficients of the system of equations, $\bf A$ and not their solution.

### 🔹 2.2.5.3 LU Decomposition vs Gauss-Jordan Elimination

At this point, those of you who are still paying attention 😜 have probably recognised that while solving a system of equation via the LU Decomposition require less operations than the Gauss-Jordan Method, the fact that you need to perform Gaussian Elimination (which is ~ 1/2 the cost of the the Gauss Jordan Method), means you probably aren't saving anything!

Let's do some accounting. For an $N \times N$ matrix

**Gauss-Jordan Elimination**:

* Gaussian Elimination to *upper triangular form* requires $\approx \frac{2}{3}N^3$ operations
* Further row reduction via backwards elimination to *reduced row-echelon form* as second step of $\approx \frac{1}{2}N^3$ operations.
* Total $\approx \frac{7}{6}N^3$ operation (not including the right-hand side) *this can change a bit given pivoting operation but $\mathcal{O}(N^3)$*.

**Gaussian Elimination and Back Substitution**:
* Gaussian Elimination to *upper triangular form* requires $\approx \frac{2}{3}N^3$ Operations
* Solution via *back substitution* $\approx N^2$ operations - often negligle compared to $N^3$
* Total $\approx \frac{2}{3}N^3$.

> ### ⚠️ Point of note
> Gauss-Jordan Elimination and Gaussian Elimination and Back Substitution
both require knowledge of the right-hand side, $\bf b$, of $\bf A\cdot x = b$, as $\bf b$ must be transformed at the same time at $\bf A$.

**LU Decomposition and Application**
* Gaussian Elimination to *upper triangular form* with $f_{ij}$ terms forming the *lower diagonal form* requires $\approx \frac{2}{3}N^3$ operations
* let $\bf L\cdot y = b$ solving for $\bf y$ via back substitution requires $\approx N^2$ operations.
* Solving $\bf U\cdot x = y$ for $\bf x$ requires $\approx N^2$ operations.

> ### ⚠️ Point of note
> For large matrices, as in the case of Gaussian-Elimination with back substitution, the cost of solving for $\bf y$ and $\bf x$ is negligible compared to the cost of the reduction to triangular forms ($\bf L$ and $\bf U$).

The **Advantage** of **LU Decomposition** is that unlike *Gauss-Jordan* and *Gaussian-Elimination with Back Substitution* the formation of $\bf L$ and $\bf U$ (the expensive part of the *LU Decomposition*), is independent of the right-hand side $\bf b$.

> ### ⚠️ Point of note
> Once computed, $\bf L, U$ can be reused to solve multiple case of $\bf A \cdot x = b$ for different value of $\bf b$ at a cost of $\mathcal{O}(N^2)$ compared to $\mathcal{O}(N^3)$ for *Gauss-Jordan* or *Gaussian Elimination*.

## 📂 2.3.6 Tridiagonal Form of $\bf A \cdot x=d$

---

In many practical engineering problems not all output $\bf d$ of a system will be influenced by all values of $\bf x$. For example, the local heat flux from a bar may only be a function of local temperature gradient (you will see more of of this when we look at solving the heat equations in or section on Ordindary Differential Equations). In such cases, the matrix $\bf A$ will be dominated by terms along the diagonal, e.g.
$$\bf A \cdot x = d$$
with
$$\bf {A} =
\begin{bmatrix}
b_1 & c_1 & 0 & 0 & \cdots & 0 \\
a_2 & b_2 & c_2 & 0 & \cdots & 0 \\
0 & a_3 & b_3 & c_3 & \cdots & 0 \\
0 & 0 & a_4 & b_4 & \cdots & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & c_{n-1} \\
0 & 0 & 0 & 0 & a_n & b_n
\end{bmatrix},
\qquad
x =
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ \vdots \\ x_{n-1} \\ x_n
\end{bmatrix},
\qquad
d =
\begin{bmatrix}
d_1 \\ d_2 \\ d_3 \\ \vdots \\ d_{n-1} \\ d_n
\end{bmatrix}.
$$

*Note*: in this notation $a_1$ and $c_n$ do not exists and are undefined.

>### 📖 Definition
> A **tridiagonal system**, as the name implies, is one that has non-zero elements on only the diagonal, plus and minus one column for a total of *three*-*tri* non-zero vectors of elements, $\bf a, b,c$ as depicted above.

This problem can be easily solved by Gaussian-Elimination and back substitution, given that the expensive process of *row reduction* now only needs to remove the $\bf a$ values.

The process to reduce just the non-zero $\bf a$ values below the diagonal requires only one row operation per column. As the rows above and including the pivot only involve two non-zero terms, each row operation will only involve two division and two subtractions. Considering each pivot point $k=1,N$.

>Rescaling the rows so that the pivot points are equal to 1 requires 2  division operations (one on values $b_k$ and one on $c_k$) for all but row $k= N$ where $c_n=0$ - *i.e. 2 operations per pivot point*.

>Reduction for pivot point $k$ requires 1 multiplication and 1 subtraction operation to reduce $a_k$ to zero. Not including the $k=N$ row which is already zero - *i.e. a further 2 operations per pivot point*.

The total operations for pivot rescaling and reduction becomes across the $M$ rows is therefore

\begin{eqnarray}
&\sum^{M-1}_{k=1} \left(2+ 2\right) + 1=\sum^{M-1}_{k=1}4 + 1
\end{eqnarray}
where the final '1' is the pivot point on row M, which only requires rescaling of $b_N$.

Recalling that $\sum^{M-1}_{k=1}1=M-1$, For square matrices ($N=M$), this reduces to
\begin{eqnarray}
&\sum^{N-1}_{k=1}& 4+1 \approx 4N,
\end{eqnarray}
not including any operations that might be required on the right-hand side of any augmented matrix.

> ### ⚠️ Point of note
> The tridiagonal nature of the system not only reduces the computation cost associated with computing the *upper diagonal* and *lower diagonal* forms, but also results in a more compact form.

Since the reduction to the upper diagonal form only requires one row operation per row and the value above the $c_i$ elements are already zero, the upper diagonal form will appear as follows

$$\bf {U} =
\begin{bmatrix}
b_1 & c_1 & 0 & 0 & \cdots & 0 \\
0 & \beta_2 & c_2 & 0 & \cdots & 0 \\
0 & 0 & \beta_3 & c_3 & \cdots & 0 \\
0 & 0 & 0 & \beta_4 & \cdots & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & c_{n-1} \\
0 & 0 & 0 & 0 & 0 & \beta_n
\end{bmatrix}$$

with the $\bf c$ elements remaining unchanged and the lower matrix of the form

$$\bf {L} =
\begin{bmatrix}
1 & 0 & 0 & 0 & \cdots & 0 \\
f_{21} & 1 & 0 & 0 & \cdots & 0 \\
0 & f_{32} & 1 & 0 & \cdots & 0 \\
0 & 0 & f_{33} & 1 & \cdots & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & 0 \\
0 & 0 & 0 & 0 & f_{n,n-1} & 1
\end{bmatrix}$$

Following *LU Decomposition*, the computational operation to solve via back subs are also signficantly reduced as now we only have to solve

$$
\begin{bmatrix}
1 & 0 & 0 & 0 & \cdots & 0 \\
f_{21} & 1 & 0 & 0 & \cdots & 0 \\
0 & f_{32} & 1 & 0 & \cdots & 0 \\
0 & 0 & f_{33} & 1 & \cdots & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & 0 \\
0 & 0 & 0 & 0 & f_{n,n-1} & 1
\end{bmatrix}
\begin{bmatrix}
y_1\\ y_2\\ y_3 \\y_4 \\ \vdots \\y_n
\end{bmatrix}
= \begin{bmatrix}
d_1\\ d_2\\ d_3 \\d_4 \\ \vdots \\d_n
\end{bmatrix}
$$

The first row of $\bf L\cdot y = d$ gives us
\begin{eqnarray}
y_1 &=& d_1\\
\end{eqnarray}
The second row becoming
\begin{eqnarray}
f_{21} y_1 + y_2 &=& d_2\\
y_2 &=& d_2-f_{21}y_1,
\end{eqnarray}
and the third
\begin{eqnarray}
f_{32} y_2 + y_3 &=& d_3\\
y_3 &=& d_3-f_{32}y_2,
\end{eqnarray}


and so on. We can express this as
\begin{equation}
y_i = d_i-f_{i,i-1}y_{i-1},
\end{equation}
which is only *one* multiplications and *one* subtraction per row. Take a second to go back and compare this to the general back-substituion in section 2.3.5.1.

We can express the total number of operations to solve for $y_i$ to $y_N$ as,
\begin{eqnarray}
\sum^N_{i=2} 2 = 2(N-1),
\end{eqnarray}
starting at $i=2$ as no calculation is required for the first row.

The solution of $\bf y$ from $\bf L$ requires $\mathcal{O}(N)$ operations. The cost of solving $\bf U \cdot x = y$ will be higher as it requires a division for non-zero diagonal, but it is still only of order $\mathcal{O}(N)$. Including the formation of the LU matrices, we are still only looking at $\approx 8N$ operations.

The solution of a *tridiagonal system* by LU Decomposition is also know as the *Thomas Algorithm*.

>### ⚠️ Point of note
>  Important Take-Away is that if you understand the composition of your problem, you can often find ways, alternate algorithms and approaches, that can significantly improve performance. This is important when you need to solve large data sets - **Big Data** problems in engineering.

>### 📖 Definition
> A **band diagonal system** is one with more non-zero terms in columns around the diagonal. While not as simple and efficient as a *tridiagonal system*, similar approach can also be used that take advantage of this. This is something you might want to consider as part of your own project and *Life Long Learning*.

---

## 📂 2.3.7 Interative Solutions to $\bf A \cdot x = b$

---

An *interative method* is one which does not seek to solve for $\bf x$ though a set number of operation directly, but instead starts with a guess at the solution and continuely refines and improves this guess until a desired accuracy is achieved. As we will see, these iterative methods can be more efficiency for large, sparse, or complex, coupled systems where direct methods become very memory-intensive.

As we mentioned earlier (section 2.3.3), the direct solutions methods will not work for near singular matrix (*Numerically Singular Matrices*) or large matrices where the summations across terms increase numerical rounding errors to a point where the values returned by the numerical implementation of the direct solution no longer satisfy the system.

Iterative methods offer a means of overcoming these limitations and improving the solution.

Let assume that we have an initial guess at the values $\bf x$, or attempts for ealier solutions, which differ from their true values by $\bf \delta x$, such that our guess is ${\bf x} + \delta \bf{x}$. Subtituting this into our systems of equations yields

\begin{align}
{\bf A}\cdot ({\bf x}+\delta {\bf x}) = {\bf b}+\delta {\bf b}\tag{2.30}
\end{align}
where $\delta \bf b$ is the change in the righthand side required to account for the non-zero $\delta \bf x$ and balance the equation. The balancing terms on the righthand side can easily be calculated given the present guess as

\begin{align}
\delta {\bf b}={\bf A}\cdot ({\bf x}+\delta {\bf x}) - {\bf b}.\tag{2.31}
\end{align}

Subtracting the equation for our sytstem $\bf A\cdot x = b$ from the equation above allows us to relate these error terms as
\begin{align}
{\bf A}\cdot \delta {\bf x} = \delta {\bf b},\tag{2.32}
\end{align}
such that we can calculate $\delta \bf x$ from $\delta \bf b$. Subtracting $\delta \bf x$ from our guess (${\bf x}+\delta \bf x$) should improve our guess. If you apply LU Decomposition, then $\bf A$ need only undergo a single Gaussian Elimination, subsequently reducing the cost of subsequent interations to improve our guess.

> ### ⚠️ Point of note
>  Solving for $\delta \bf x$ is as expensive as directly solving for $\bf x$, but can be used where direct methods have failed!




## 📂 2.3.8 The Jacobi Method for Iterative Solution of $\bf A \cdot x = b$

---

The **Jacobi Method**, is an iterative method that allows you to start from an approximation to the true solution and obtain better and better approximations from a computation cycle until a desired accuracy is achieved. This methods avoids the need to reduce the $\bf A$ to a *lower echelon form* and potentially allows you to balance the computation demand with the required degree of accuracy.

To help explain this, it helps to recall that solving $\bf A \cdot x = b$,

\begin{equation}
\begin{bmatrix}
a_{11} & a_{12} & a_{13}\\
a_{21} & a_{22} & a_{23}\\
a_{31} & a_{32} & a_{33}
\end{bmatrix}
\begin{bmatrix}
x_{1}\\
x_{2}\\
x_{3}
\end{bmatrix}
=
\begin{bmatrix}
b_{1}\\
b_{2}\\
b_{3}
\end{bmatrix}
\end{equation}

requires solving a series of simultaneous equations, i.e.

\begin{aligned}
a_{11}x_1 + a_{12}x_2 + a_{13}x_3 &= b_1 \\
a_{21}x_1 + a_{22}x_2 + a_{23}x_3 &= b_2 \\
a_{31}x_1 + a_{32}x_2 + a_{33}x_3 &= b_3
\end{aligned}

We can rearrage each of these equations and express them in terms of the target unknown $\bf x$, i.e.

\begin{aligned}
x_1 &= \frac{1}{a_{11}}\left(b_1- a_{12}x_2 - a_{13}x_3\right)  \\
x_2 &= \frac{1}{a_{22}}\left(b_2 - a_{21}x_1 - a_{23}x_3\right)\\
x_3 &= \frac{1}{a_{33}}\left(b_3 - a_{31}x_1 - a_{32}x_2\right)
\end{aligned}

where in each case, if we knew the other values of $\bf x$ we could compute the current value $x_i$.

Hence, if we have a guess for $\bf x$ it is easy to calculate a value of $x_i$ for each of the i rows of the matrix or, in other words, each equation. If our guess at $\bf x$ is correct then the left and right hands side of these equation will balance. Easy!

Let's introduce a superscript to represent the iteration number associated with the values of $\bf x$, such that $x^m_i$ represents the i-th value of the $\bf x$ after $m$ iterations. Defining $x^0_i$ as our initial guess, we compute the first update to our solution as

\begin{aligned}
x^1_1 &= \frac{1}{a_{11}}\left(b_1- a_{12}x^0_2 - a_{13}x^0_3\right)  \\
x^1_2 &= \frac{1}{a_{22}}\left(b_2 - a_{21}x^0_1 - a_{23}x^0_3\right)\\
x^1_3 &= \frac{1}{a_{33}}\left(b_3 - a_{31}x^0_1 - a_{32}x^0_2\right)
\end{aligned}

and use the updated values of $\mathbf{x}$ as input into our second iteration as

\begin{aligned}
x^2_1 &= \frac{1}{a_{11}}\left(b_1- a_{12}x^1_2 - a_{13}x^1_3\right)  \\
x^2_2 &= \frac{1}{a_{22}}\left(b_2 - a_{21}x^1_1 - a_{23}x^1_3\right)\\
x^2_3 &= \frac{1}{a_{33}}\left(b_3 - a_{31}x^1_1 - a_{32}x^1_2\right)
\end{aligned}

>### ⚠️ Point of note
> In this case $x^2$ is not the square of $x$! It is value computed from the second iteration.

After $m$ iterations, we can express the next iteration as
\begin{aligned}
x^{m+1}_1 &= \frac{1}{a_{11}}\left(b_1- a_{12}x^m_2 - a_{13}x^m_3\right)  \\
x^{m+1}_2 &= \frac{1}{a_{22}}\left(b_2 - a_{21}x^m_1 - a_{23}x^m_3\right)\\
x^{m+1}_3 &= \frac{1}{a_{33}}\left(b_3 - a_{31}x^m_1 - a_{32}x^m_2\right)
\end{aligned}

For a general NxN matrix, the update to values of $\bf x$ can be expressed as

$$x^{m+1}_i = \frac{1}{a_{ii}}\left(b_i- \sum^{i-1}_{j=1} a_{ij}x^m_j - \sum^N_{j=i+1}a_{ij}x^m_j\right) \tag{2.33}$$

This can be expressed in terms of matrix operations if you think about the matrix $\bf A$ as the some of its upper, lower and diagonal components, i.e.
$$ \bf A = D+L+U \tag{2.34}$$
where
$$ A =
\begin{bmatrix}
a_{11} & a_{12} & a_{13} & a_{14} \\
a_{21} & a_{22} & a_{23} & a_{24} \\
a_{31} & a_{32} & a_{33} & a_{34} \\
a_{41} & a_{42} & a_{43} & a_{44}
\end{bmatrix}
$$

$$ D =
\begin{bmatrix}
a_{11} & 0 & 0 & 0 \\
0 & a_{22} & 0 & 0 \\
0 & 0 & a_{33} & 0 \\
0 & 0 & 0 & a_{44}
\end{bmatrix},
L =
\begin{bmatrix}
0 & 0 & 0 & 0 \\
a_{21} & 0 & 0 & 0 \\
a_{31} & a_{32} & 0 & 0 \\
a_{41} & a_{42} & a_{43} & 0
\end{bmatrix},
U =
\begin{bmatrix}
0 & a_{12} & a_{13} & a_{14} \\
0 & 0 & a_{23} & a_{24} \\
0 & 0 & 0 & a_{34} \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

>### ⚠️ Point of note
> This is not the LU Decomposition from before. As this is a sum and not a product of matrices, this does not require any operation on these values, i.e. $a_{12}$ in $\bf U$ is the same value, in the same locations as in $\bf A$.

The iterative update of $\bf x^{m+1}$ can then be expressed as

$$ {\bf x}^{m+1} = {\bf D}^{-1}\left[{\bf b - L x}^m - {\bf U x}^m\right] \tag{2.35}$$

and computed as matrix operations, rather than having to loop through values of $i$ - *Can you think why matrix multiplication is probably a better way to solve this?*

**When do we stop iterating?**

In general, you want to continue iterating until the solution stops changing. Numerically, this will never be better than the machine precision. We can define the relative magnitude of the change in each value of $\bf x$ as

$$ |\epsilon_{a,i}| = \left| \frac{x^{m+1}_i-x^m_i}{x^{m+1}_i}\right| < \epsilon_s \tag{2.36}$$

within a target or machine precision of $\epsilon_s$.

There is a possibility that an iterative solution will fail to converge to the desired accuracy or may diverge if it is numerically unstable, in which case $\epsilon_{a,i}$ will begin to increase.

The *Jacobi Method* will always converge if it is *Strictly Diagonally Dominant*, i.e. when the terms on the diagonal of the coefficient matrix $\bf A$ and greater than the sum of the magnitude of the other terms,

$$|a_{ii}| \ge \sum^N_{j=1, i\ne j}|a_{ij}| \tag{2.37}$$.

Some non-diagonally dominant systems may still be stable, but stability is not guaranteed in those cases.

**We will return to concepts of numerical stability later in the unit.**

There are alternative iterative methods that provide faster convergence than the Jacobi, but may be less stable. As is almost always the case, you need to consider these tradeoff between methods.

If your project requires the solution of an linear system, I encourage you to search for other possible iterative methods and consider/test who these might match your problem.

## 📂 2.3.9 Singular Value Decomposition (SVD) for solving $\bf A \cdot x = b$

---

Consider our linear system as we did for the iterative improvement in section 2.3.7, but rather than imposing an error in $\bf x$, let us instead consider an offset to the right-hand side of the equation $\delta \bf b$,

$${\bf A \cdot x} ={\bf b} + \delta{\bf b}$$

Multiplying both sides by $\bf A^{-1}$ allows us to express our solution to this system as

$${\bf x} ={\bf A}^{-1}{\bf b} + {\bf A}^{-1}\delta{\bf b}$$

The change and hence the error in $\bf x$, due to the offset in the solution corresponds to ${\bf A}^{-1}\delta{\bf b}$.

It turns out we can use the singular values of a matrix to tell us which changes in $\bf b$ will give the largest errors in $\bf x$ and in doing so help us arrive at a solution for $\bf x$.

It's also really useful when trying to diagnose why your attempt at Gaussian Elimination or LU Decomposition might have failed. Infact, its the method of choice for most *linear least-squares problems*, in other words, its really useful!

**What do we mean by Singular Value Decomposition (SVD)?**

If $\bf A$ is a matrix of $M$ rows and $N$ columns ($M\times N$), then as long as $M \ge N$ (more equations than unknowns) we can express $\bf A$ as the product of an $M\times N$ **column-orthogonal** matrix $\bf U$, a $N\times N$ **diagonal matrix** $\bf \Sigma$, and the transpose of an $N \times N$ **orthogonal matrix** V.

$$ {\bf A = U\cdot \Sigma \cdot V}^T$$

where

\begin{eqnarray}
\mathbf{U} &=&
\begin{bmatrix}
u_{11} & u_{12} & u_{13} & u_{14} \\
u_{21} & u_{22} & u_{23} & u_{24} \\
u_{31} & u_{32} & u_{33} & u_{34} \\
u_{41} & u_{42} & u_{43} & u_{44}
\end{bmatrix},
\quad {\bf U}^T\cdot {\bf U} = {\bf I} \\
\mathbf{\Sigma} &=&
\begin{bmatrix}
\sigma_{1} & 0 & 0 & 0 \\
0 & \sigma_{2} & 0 & 0 \\
0 & 0 & \sigma_{3} & 0 \\
0 & 0 & 0 & \sigma_{4}
\end{bmatrix},
\quad
\sigma_{1} \ge \sigma_{2} \ge \sigma_{3} \ge \sigma_{4} \ge 0,\\
\mathbf{V}^{T} &=&
\begin{bmatrix}
v_{11} & v_{21} & v_{31} & v_{41} \\
v_{12} & v_{22} & v_{32} & v_{42} \\
v_{13} & v_{23} & v_{33} & v_{43} \\
v_{14} & v_{24} & v_{34} & v_{44}
\end{bmatrix},
\quad
\mathbf{V}^{T}\mathbf{V} = \mathbf{I}
\end{eqnarray}

>### 📖 Definition
> An **orthgonal matrix** is one whose transpose, e.g. ${\bf U}^T$ is equal to its inverse, ${\bf U}^{-1}$, such that the product of the matrix with its transpose is equal to the identity matrix. The derivation of this is beyond the scope of this unit.

Importantly, for us, the diagonal values $\sigma_i$, which can be positive or zero, are the *singular value* of the origonal matrix $\bf A$.

> ### ⚠️ Point of note
> It turns out that even though a unique SVD only exists when $M \ge N$, we can perform a SVD on a matrix where $M < N$ (more unknowns that equations), but the solution will not be unique. More on this later.
>
>See https://sites.ecse.rpi.edu/~qji/CV/svd_review.pdf

### 🔹 2.3.9.1 SVD of a Square $N \times N$ Matrix

To help understand how we can undertake and use a SVD, let's first consider a square $N \times N$ matrix $\bf A$. In this case, $N = M$ and $\bf U, \Sigma, V$ are all $N \times N$ square matrices. Recall that

$$ {\bf A = U\cdot \Sigma \cdot V^T} \tag{2.38}$$

and the inverse of the product of two matrices is given by $({\bf A\cdot B})^{-1} = {\bf B}^{-1}\cdot {\bf A}^{-1}$ such that

$$ {\bf A}^{-1} = {\bf V\cdot \Sigma^{-1} \cdot U}^T, \tag{2.39}$$

since ${\bf U}^{-1} = {\bf U}^{T}$ and $\left({\bf V}^T\right)^{-1} = {\bf V}$, given that both matrices are *orthogonal* - see section 2.3.9 for definition.

This brings us to the first significant value of an SVD. If any singular value $\sigma_{ii}$ is sufficiently close to zero, then the corresponding terms of ${\bf \Sigma}^{-1}$. i.e. $1/\sigma_{ii}$ will be infinite and we can see we will have a problem inverting this system. The more value of $\sigma_{ii}$ that present this problem, the more *singular* it is.

>### 📖 Definition
> The **condition number** of a matrix is defined as the ratio of the magnitude of the largest value of $\sigma_{ii}$ to the smallest,
>
>$$ \mbox{cond}({\bf A})=\frac{\mbox{Max}(|\sigma_{ii}|)}{\mbox{Min}(|\sigma_{ii}|)}, \tag{2.40}$$

An infinite cond($\bf A$) indicates a **singular matrix**, with an *ill-conditioned* matrix having a cond($\bf A$) $\ge 10^6$ for a single precision matrix or cond($\bf A$) $\ge 10^{12}$ for double precision.

>### ⚠️ Point of note
> These *ill-conditioned* matrices represent systems that are effectively numerically singular relative to machine precision.

Now consider how we can use the inverse of the SVD form to solve $\bf A\cdot x = b$. Recall the direct solution is given by ${\bf x = A}^{-1}\bf b$, which in SVD form is given by

$$ {\bf x} = {\bf V\cdot \Sigma}^{-1} \cdot \left({\bf U}^T\cdot {\bf b}\right),\tag{2.41}$$

which for a 4x4 matrix can be written out as

\begin{align}
\begin{bmatrix}
x_1\\
x_2\\
x_3\\
x_4
\end{bmatrix}=
\begin{bmatrix}
v_{11} & v_{12} & v_{13} & v_{14} \\
v_{21} & v_{22} & v_{23} & v_{24} \\
v_{31} & v_{32} & v_{33} & v_{34} \\
v_{41} & v_{42} & v_{43} & v_{44}
\end{bmatrix}
⋅\begin{bmatrix}
1/\sigma_{1} & 0 & 0 & 0 \\
0 & 1/\sigma_{2} & 0 & 0 \\
0 & 0 & 1/\sigma_{3} & 0 \\
0 & 0 & 0 & 1/\sigma_{4}
\end{bmatrix}
⋅\begin{bmatrix}
u_{11} & u_{21} & u_{31} & u_{41} \\
u_{12} & u_{22} & u_{32} & u_{42} \\
u_{13} & u_{23} & u_{33} & u_{43} \\
u_{14} & u_{24} & u_{34} & u_{44}
\end{bmatrix}⋅
\begin{bmatrix}
b_1\\
b_2\\
b_3\\
b_4
\end{bmatrix}
\end{align}

where

$$x_k = \sum^N_{i=1}\left[\frac{v_{ki}}{\sigma_i}\sum^N_{j=1} \left(u_{ji}b_j\right)\right]$$

If ${\bf b} =0$ then SVD shows us that the only possible solution for a well conditioned (non-signular matrix) is ${\bf x} = 0$.

Now consider what happens if $\bf A$ is a singular matrix, for example let $\sigma_1 = 0$. The term  $v_{k1}/\sigma_1$ is not undefined for all values of $x_k$. This means ${\bf A \cdot x} = 0$ can now have an infinite series of solutions as $x_k$ is nolonger uniquely defined by  $\bf b$.

In fact, any column $j$ of $\bf V$, i.e. whose corresponding singular value $\sigma_j$ is zero is a solution.

**You don't believe me?** Consider the following example.



### 🧪 Example of a Singular Matrix and Its Solutions Space

Consider this example where the SVD of a 3x3 matrix is given by
$$ {\bf U} =
\begin{bmatrix}
0 & 1 & 0\\
1 & 0 & 0\\
0 & 0 & 1
\end{bmatrix},
\qquad
{\bf V} =
\begin{bmatrix}
0 & 1 & 0\\
1 & 0 & 0\\
0 & 0 & 1
\end{bmatrix}.
$$

Both satisfy the orthogonality, $ {\bf U}^{T}{\bf U} = \bf I $, ${\bf V}^{T}{\bf V} = \bf I $.

We can make this a singular by defining $\bf \Sigma$ as

$$ \mathbf{\Sigma} =
\begin{bmatrix}
2 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 0
\end{bmatrix}.
$$

You can show this corresponds to the coefficient matrix

$$ {\bf A} =
\begin{bmatrix}
1 & 0 & 0\\
0 & 2 & 0\\
0 & 0 & 0
\end{bmatrix}
= {\bf U\cdot \Sigma \cdot V}^{T},
$$

> ### 🗣️ AI Prompts
> Gemini (July 2026) was used to generate this example with prompts 'create an example with simpile integer values in U V Sigma space for a singular matrix'. This was done to avoid the floating point values that often appear when computing the SVD form.

Let's solve the homogeneous system  ${\bf A\cdot x} = 0$, where
$${\bf x} =
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}.
$$

We can expand $\bf A \cdot x$ as

$$ {\bf A\cdot x} =
\begin{bmatrix}
1 & 0 & 0\\
0 & 2 & 0\\
0 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}
=
\begin{bmatrix}
x_1\\
2x_2\\
0
\end{bmatrix}
=
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}.
$$

Our system of linear equations is therefore:
\begin{eqnarray}
x_1 = 0 \\
2x_2 = 0 \Rightarrow x_2 = 0
\end{eqnarray}

But this tells us nothing about $x_3$! The third row gives $0 = 0$ (no constraint on $x_3$).

Let's say that $x_3 = t$. Given we found $x_1 = x_2 = 0$, $\bf x$ becomes

\begin{eqnarray}
{\bf x} &=&
\begin{bmatrix}
0\\
0\\
t
\end{bmatrix}\\
{\bf x} &=& t
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix},\quad t \in \mathbb{R}.
\end{eqnarray}

There are infinitely many solutions to ${\bf A\cdot x} = 0 $, one for each real $t$.

Let's consider this in terms of our SVD of $\bf A$, where $\mathbf{x} = \mathbf{V}\cdot \mathbf{\Sigma}^{-1}\cdot\left(\mathbf{U}^T\cdot \mathbf{b}\right)$.
\begin{align}
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}=
\begin{bmatrix}
v_{11} & v_{12} & v_{13}  \\
v_{21} & v_{22} & v_{23}  \\
v_{31} & v_{32} & v_{33}  \\
\end{bmatrix}
⋅\begin{bmatrix}
1/\sigma_{1} & 0 & 0 \\
0 & 1/\sigma_{2} & 0 \\
0 & 0 & 1/\sigma_{3} \\
\end{bmatrix}
⋅\begin{bmatrix}
u_{11} & u_{21} & u_{31} \\
u_{12} & u_{22} & u_{32} \\
u_{13} & u_{23} & u_{33}
\end{bmatrix}⋅
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
\end{align}

which for the present examples becomes

\begin{align}
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}=
\begin{bmatrix}
0 & 1 & 0  \\
1 & 0 & 0  \\
0 & 0 & 1  \\
\end{bmatrix}
⋅\begin{bmatrix}
1/2 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1/0 \\
\end{bmatrix}
⋅\begin{bmatrix}
0 & 1 & 0 \\
1 & 0 & 0 \\
0 & 0 & 1
\end{bmatrix}⋅
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
\end{align}

and reduces to

\begin{align}
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}=
\begin{bmatrix}
0 & 1 & 0 \\
1/2 & 0 & 0 \\
0 & 0 & 1/0 \\
\end{bmatrix}
⋅\begin{bmatrix}
0 & 1 & 0 \\
1 & 0 & 0 \\
0 & 0 & 1
\end{bmatrix}⋅
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
\end{align}

\begin{align}
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}=
\begin{bmatrix}
1 & 0 & 0 \\
0 & 1/2 & 0 \\
0 & 0 & 1/0 \\
\end{bmatrix}⋅
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
\end{align}

hence,
\begin{eqnarray}
x_1 &=& 0 \\
x_2 &=& (1/2)0 ⇒ x_2 =0\\
x_3  &=& ?
\end{eqnarray}

This matches the SVD. The zero singular value in $\bf \Sigma$ corresponds to infinite solutions for the system.

Returning to our previous statement in section 2.3.9.1 that
>any column $j$ of $\bf V$, i.e. whose corresponding singular value $\sigma_j$ is zero is solution.
since $\sigma_3 = 0$, then $\mathbf{x}$ is equal to the 3rd column of $\mathbf{V}$, i.e.
$$ {\bf x} = \begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix} = v_{i3} =
\begin{bmatrix}
v_{13}\\
v_{23}\\
v_{33}\\
\end{bmatrix}
= \begin{bmatrix}
0\\
0\\
1\\
\end{bmatrix}
$$

Which is consistent with solution for $\bf x$ from above.


### 🔹 2.3.9.2 Consider a Singular System when $\bf b\ne 0$

When ${\bf b} \ne 0$, the singular nature of the system means we have infinite solutions, not just a series of scale solution like we had for $\mathbf{b} = 0$. Taking our example from before,

We can expand $\bf A \cdot x$ as

$$ {\bf A\cdot x} =
\begin{bmatrix}
1 & 0 & 0\\
0 & 2 & 0\\
0 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}
=
\begin{bmatrix}
x_1\\
2x_2\\
0
\end{bmatrix}
=
\begin{bmatrix}
b_1\\
b_2\\
b_3
\end{bmatrix}.
$$

Our system of linear equations is therefore:
\begin{eqnarray}
x_1 &=& b_1 \\
2x_2 &=& b_2 \Rightarrow x_2 = b_2/2,
\end{eqnarray}

but again, this tells us nothing about $x_3$, so it can take any value and still be a valid solution.

For a reason we will return to in a minute, let's assume we want the smallest length vector in $\bf x$, i.e. smallest $|{\bf x}|$.

From the SVD we showed (equation 2.41) that

$$ {\bf x} = {\bf V\cdot \Sigma}^{-1} \cdot \left({\bf U}^T\cdot {\bf b}\right),$$

which corresponds to

$$x_k = \sum^N_{i=1}\left[\frac{v_{ki}}{\sigma_i}\sum^N_{j=1} \left(u_{ji}b_j\right)\right]$$

given that the $x_k$ value associated with the singular values $\sigma_k$ can take any values, the smallest magnitude value it can take is zero. This can be achieved by replacing the value $1/\sigma_k$ with 0.

> ### ⚠️ Point of note
> This has implications for least-squared solutions to linear system, see next section.

### 🔹 2.3.9.3 What Happens When There is No Solution for $\bf x$?

There is no guarentee that a solution exist for all cases of $\bf A\cdot x = b$. The good news it that we can still use SVD to approximate a solution via equation 2.41, i.e.

$$ {\bf x} = {\bf V\cdot \Sigma}^{-1} \cdot \left({\bf U}^T\cdot {\bf b}\right),$$

If as in the previous section we set $1/\sigma_k=0$ where $\sigma_k = 0$. We can setup equations to solve for $\mathbf{x}$ as we did in 2.3.9.3.

In such cases this will not be an exact solution for $\bf A \cdot x = b$, but will give the closest possible solution in a *least squares* case, i.e.

$$ {\bf x}\quad\mbox{ will minimise }\quad r = |{\bf A\cdot x-b}|$$

where $r$ is the residual of the solution.

The proof for this gets beyond the scope of this unit, but can be explain through the recognition that an SVD is effectively a rotation by ${\bf V}^T$, and scaling in the transformed coordinates by the singular values of $\bf \Sigma$ following by a final rotation $\bf U$ - similar to way matrix multiplication is directly used for rotations as discussed in section 2.3.1.

The role of $\bf V$ in $ {\bf x} = {\bf V\cdot \Sigma}^{-1} \cdot \left({\bf U}^T\cdot {\bf b}\right)$ is similar a rotation of the solution vector $\bf y$ where $\bf y={\bf \Sigma}^{-1}\cdot{\bf U}^T\cdot {\bf b}$ and
\begin{align} y_i &= \frac{u^T_i {\bf b}}{\sigma_i} \quad&\mbox{ for }\sigma_i \ne 0\\
y_i &= 0 \quad&\mbox{ for }\sigma_i = 0
\end{align}  
where $u^T_i$ is the i-th row of the matrix $\bf {U}^T$.

By setting $y_i=0$ where $\sigma_i = 0$ we replace the infinite possible solutions for this value of $\bf x$ to that which has the minimal value before it is projected back to the solution space by $\bf V$.

> ### ⚠️ Point of note
>  The other advanges is that this can be applied even when the number of equation is not equal to the number of variables!






### 🧪 Example of a Least Square Solution via SVD

Consider a system of linear equations, with the following singular values.

$$ {\bf U} = \begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1\\
\end{bmatrix}
\quad {\bf \Sigma} = \begin{bmatrix}
1 & 0\\
0 & 1 \\
0 & 0 \\
\end{bmatrix}
\quad {\bf V}^T = \begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}$$

which from ${\bf A} = {\bf U}\cdot{\bf \Sigma}\cdot{\bf V}^T$

\begin{align}
{\bf A}
&= \begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1\\
\end{bmatrix}⋅
 \begin{bmatrix}
1 & 0\\
0 & 1 \\
0 & 0 \\
\end{bmatrix}⋅
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}\\
{\bf A}
&= \begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1\\
\end{bmatrix}⋅
\begin{bmatrix}
1 & 0\\
0 & 1\\
0 & 0\\
\end{bmatrix}\\
{\bf A}&= \begin{bmatrix}
1 & 0\\
0 & 1\\
0 & 0\\
\end{bmatrix}
\end{align}

Expressed as a linear system, ${\bf A\cdot x} = \bf b$, this is a system if two unknowns and three variables and is a

\begin{align}
\begin{bmatrix}
1 & 0\\
0 & 1\\
0 & 0\\
\end{bmatrix}⋅\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix} = \begin{bmatrix}
b_1\\
b_2\\
b_3
\end{bmatrix}
\end{align}

This is obviously a fairly trivial example, but we can use this to help demonstrate the process. Let's assume ${\bf b}= [1\mbox{ }2\mbox{ }3]^T$.

This is not a square matrix (number of equations is greater than the number if unknowns), so we cannot invert this 3x2 matrix, but we can use a SVD of the system to solve for $\bf x$. From our SVD we showed that

$$ {\bf x} = {\bf V\cdot \Sigma}^{-1} \cdot \left({\bf U}^T\cdot {\bf b}\right),$$

which we can evaluate as

\begin{align}
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix} &=
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}⋅
\begin{bmatrix}
1 & 0\\
0 & 1\\
0 & 0
\end{bmatrix}^{-1}⋅
\begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
0 & 0 & 1
\end{bmatrix}⋅
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}\\
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix} &=
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}⋅
\begin{bmatrix}
1 & 0\\
0 & 1\\
0 & 0
\end{bmatrix}^{-1}⋅
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}
\end{align}

Here, $\bf \Sigma$ isn't square so we can't compute a true inverse, but seeing as we are going to replace $1/0$ with $0$, we can use the pseudo-inverse, i.e.

$$ {\bf \Sigma}^{-1}= \begin{bmatrix}
\sigma_1 & 0\\
0 & \sigma_2\\
0 & 0
\end{bmatrix}^{-1} \\
 {\bf \Sigma}^+ = \begin{bmatrix}
\frac{1}{\sigma_1} & 0 & 0\\
0 & \frac{1}{\sigma_2} & 0\\
\end{bmatrix}$$

> ### ⚠️ Point of note
> **Pseudo-inverse* involves a transpose for non-square matrices.

The Least Square solution then becomes

\begin{align}
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix} &=
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}⋅
\begin{bmatrix}
1 & 0 & 0\\
0 & 1 & 0\\
\end{bmatrix}⋅
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}\\
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix} &=
\begin{bmatrix}
1 \\
2
\end{bmatrix}
\end{align}

> ### ⚠️ Point of note
> We can use this SVD solution approach to get the best possible solution to linear systems, when have more equation than unknowns and our system is *overdetermined*, or extract one of an infinite number of possible solutions (corresponding to the smallest $|\bf x|$) when we have less equations than unknowns and our system is *underdetermined*.


## 📂 2.3.10 Performing a Single Value Decomposition

---

I you think back to ENG2005 and some of things you covered there, you might have thought they would never be relevant to Mechanical/Aerospace engineering. Well, I am sorry to tell you, thats not the case.

Hopefully the idea of **eigenvalues** might ring a little bell. The application here is pretty straight forward.

Let's start with a short refresher on a couple of matrix concepts.

> ### 📖 Definition
>**Determinant of a matrix:**
>
>   for a matrix
>   $$ {\bf A} = \begin{bmatrix}
   a_{11} & a_{12} & a_{13}\\
   a_{21} & a_{22} & a_{23}\\
   a_{31} & a_{32} & a_{33}
   \end{bmatrix}$$
   the determinate
   $$ |{\bf A}| = \begin{vmatrix}
   a_{11} & a_{12} & a_{13}\\
   a_{21} & a_{22} & a_{23}\\
   a_{31} & a_{32} & a_{33}
   \end{vmatrix}$$
   can be expressed as the sum of the product of each term along any row or column, multiplied by the determinate of the matrix, with the row and column that contain that term removed. For example, if we consider each term of the first row of $\bf A$, the determinant can be expressed as
   $$|{\bf A}| = a_{11}\begin{vmatrix}
   a_{22} & a_{23} \\
   a_{32} & a_{33}\end{vmatrix}
   -a_{12}\begin{vmatrix}
   a_{21} & a_{23} \\
   a_{31} & a_{33}\end{vmatrix}
   +a_{13}\begin{vmatrix}
   a_{21} & a_{22} \\
   a_{31} & a_{32}\end{vmatrix}
   $$
   where the determinant of a 2x3 matrix is simply the product of the terms on the diagonal minus the product of terms on the off-diagonal, i.e.
   $$\begin{vmatrix}
   a_{22} & a_{23} \\
   a_{32} & a_{33}\end{vmatrix} = a_{22}a_{33}-a_{23}a_{32}
   $$
  This process can be repeated for large matrices, each step reducing the size of the determinats till you get to 2x2.
>
>**Eigenvalues of a matrix**
>
>  Every matrix is characterised by a set of **eigenvector**, which represent the principal, invariant directions of a linear transformation of $\mathbf{A}$ or how coefficient of a system map the variables $\mathbf{x}$ to the output $\mathbf{b}$. The **eigenvalues** represent the scaling of these vectors.
>
>  The eigenvalue $\lambda$ of $\bf A$ are calculated from
  $$ |{\bf A}-\lambda{\bf I}| = 0$$
  The determinant on the left-hand side represents the characteristic polynomial in terms of $\lambda$, the roots of which represent the *eigenvalues*. E.g.
  $$ {\bf A} = \begin{bmatrix}
   a_{11} & a_{12} \\
   a_{21} & a_{22}
   \end{bmatrix}$$
  then
  $$|{\bf A}-\lambda{\bf I}| = \begin{vmatrix}
  a_{11}-\lambda & a_{12}\\
   a_{21} & a_{22}-\lambda
  \end{vmatrix}=(a_{11}-\lambda)(a_{22}-\lambda)-a_{12}a_{21}=0$$
>
>The eigenvector can be calculated from
  $$({\bf A} − \lambda_i {\bf I}){\bf v}_i = 0.$$
>
>If the first eigenvalue is $\lambda_1$ then the corresponding eigenvector must
>    
>  $$\begin{bmatrix}
   a_{11}-\lambda_1 & a_{12} \\
   a_{21} & a_{22}-\lambda_1
   \end{bmatrix}\cdot\begin{bmatrix}
   v_{\lambda_1,1}\\
   v_{\lambda_1,2}
   \end{bmatrix} = 0$$

So how does this helps us get to a Singular Value Decomposition?

It turns out that:
* The singular value in $\bf W$ are $\sigma_i = \sqrt{\lambda_i}$, where $\lambda_i$ are the eigenvalues of ${\bf A}^T {\bf A}$
* The unit eigenvectors of ${\bf A}^T {\bf A}$, i.e. ${\bf \hat{v}}_i$ are the columns of ${\bf V}$
* The value of $\bf U$ can be computed as
  $${\bf u}_i = \frac{1}{\sigma_i}{\bf A}{\bf \hat{v}}_i$$
  where ${\bf u}_i$ are columns of $\bf U$.

For larger matrices, the efficient SVD is a topic of its own, with methods for very large matrices, focusing on recovering only the largest and most dominant singular values for a lower-ranked approximation to the problem. We will show why this is practical in our workshops and practicals.

### 🧪 Example of SVD of ${\bf A}$

Consider the matrix
$${\bf A} =\begin{bmatrix}
2 & 0 & 0\\
2 & 1 & 0\\
0 &-2 & 0
\end{bmatrix}$$

1. Calculate ${\bf A}^T{\bf A}$.

\begin{align}
{\bf A}^T{\bf A} &=\begin{bmatrix}
2 & 2 & 0\\
0 & 1 & -2\\
0 & 0 & 0\end{bmatrix}⋅
\begin{bmatrix}
2 & 0 & 0\\
2 & 1 & 0\\
0 &-2 & 0
\end{bmatrix}\\
{\bf A}^T{\bf A} &=
\begin{bmatrix}
8 & 2 & 0\\
2 & 5 & 0\\
0 & 0 & 0
\end{bmatrix}
\end{align}

2. Find the *eigenvalues* of ${\bf A}^T{\bf A}$.

    \begin{align}
  |{\bf A}^T{\bf A}-\lambda{\bf I}| = \begin{vmatrix}
  8-\lambda & 2 & 0\\
  2 & 5-\lambda & 0\\
  0 & 0 & -\lambda
  \end{vmatrix} = 0
  \end{align}

    The easiest way to calculate the deteminant is to consider the row or column with the most zeros, so if we express the determinant in terms of the bottom row we get

    \begin{align}
|{\bf A}^T{\bf A}-\lambda{\bf I}| &= -\lambda\begin{vmatrix}
8-\lambda & 2\\
2 & 5-\lambda
\end{vmatrix} = -\lambda\left[(8-\lambda)(5-\lambda)-4 \right] = 0\\
&=-\lambda\left(\lambda^2 -13\lambda +36 \right)=0
\end{align}

    For the determinant zero, either $\lambda = 0$ or $\lambda^2 -13\lambda +36 =0$

    \begin{align}
\lambda &= \frac{13\pm\sqrt{13^2-4\times36}}{2}\\
\lambda &= \frac{13\pm 5}{2}
\end{align}

    resulting in the *eigenvalues* $\lambda_1 = 9$, $\lambda_2 = 4$, $\lambda_3 = 0$.
    
    By convention we will rank them in descending order of magnitude (from largest to smallest). This convention is important for when we convert them to singular values and as it represents the relative impact of the corresponding eigenvector and has implication in SVD (more on that in a second).

3. Find the corresponding *eigenvectors* of ${\bf A}^T{\bf A}$

    * for $\lambda_1 = 9$

        \begin{align}
    ({\bf A}^T{\bf A} − 9 {\bf I}){\bf v}_1 =
\begin{bmatrix}
-1 & 2 & 0\\
2 & -4 & 0\\
0 & 0 & -9\end{bmatrix}⋅
\begin{bmatrix}
v_1\\
v_2\\
v_3
\end{bmatrix} = 0
    \end{align}

        We can use Gaussian Elimination to solve for $\bf v$,

        \begin{align}
    &\left[
    \begin{array}{ccc|c}
    -1 & 2 & 0 &0\\
    2 & -4 & 0 & 0\\
    0 & 0 & -9 & 0
    \end{array}\right]\\
    R'_2 \leftarrow R_2+2R_1
    &\left[
    \begin{array}{ccc|c}
    -1 & 2 & 0 &0\\
    0 & 0 & 0 & 0\\
    0 & 0 & -9 & 0
    \end{array}\right]
    \end{align}

        hence
    
        \begin{align}
    \begin{bmatrix}
    -1 & 2 & 0\\
    0 & 0 & 0\\
    0 & 0 & -9\end{bmatrix}⋅
    \begin{bmatrix}
    v_1\\
    v_2\\
    v_3
    \end{bmatrix} &= 0\\
    -v_1 + 2v_2 &= 0\\
    -9v_3 &= 0
    \end{align}

        This yields $v_3 = 0$ and $v_1 =2v_2$ but $v_2$ can be anything.
        
        Since we need to normalise the unit vectors by their length, we can any value for $v_2$, other than zero which would be a null eigenvector.

        Say we take $v_2 = 1$, ${\bf v}_1 = (2, 1, 0)^T$, with a corresponding unit vector

        \begin{align}\hat{\bf v}_1 = \frac{\bf v}{|\bf v|}=\frac{1}{\sqrt{5}}
        \begin{bmatrix} 2 \\ 1 \\ 0\end{bmatrix}\end{align}

    * for $\lambda_2 = 4$

        \begin{align}
        ({\bf A}^T{\bf A} − 4 {\bf I}){\bf v}_2 =
        \begin{bmatrix}
        4 & 2 & 0\\
        2 & 1 & 0\\
        0 & 0 & -4\end{bmatrix}⋅
        \begin{bmatrix}
        v_1\\
        v_2\\
        v_3
        \end{bmatrix} = 0
        \end{align}

        By Gaussian Elimination

        \begin{align}
    &\left[
    \begin{array}{ccc|c}
    4 & 2 & 0 &0\\
    2 & 1 & 0 & 0\\
    0 & 0 & -4 & 0
    \end{array}\right]\\
    R'_2 \leftarrow R_2+0.5R_1
    &\left[
    \begin{array}{ccc|c}
    4 & 2 & 0 &0\\
    0 & 0 & 0 & 0\\
    0 & 0 & -4 & 0
    \end{array}\right]
    \end{align}

        hence
        \begin{align}
    4v_1 + 2v_2 &= 0\\
    -4v_3 &= 0
    \end{align}

        Similar to above if we take $v_1 = 1$, ${\bf v}_2 = (1, -2, 0)^T$, with a corresponding unit vector

        \begin{align}\hat{\bf v}_2 = \frac{\bf v}{|\bf v|}=\frac{1}{\sqrt{5}}
        \begin{bmatrix} 1 \\ -2 \\ 0\end{bmatrix}\end{align}

    * for $\lambda_3 = 0$

        \begin{align}
    ({\bf A}^T{\bf A} − 0 {\bf I}){\bf v}_3 =
\begin{bmatrix}
8 & 2 & 0\\
2 & 5 & 0\\
0 & 0 & 0\end{bmatrix}⋅
\begin{bmatrix}
v_1\\
v_2\\
v_3
\end{bmatrix} = 0
    \end{align}

        By Gaussian Elimination

        \begin{align}
    &\left[
    \begin{array}{ccc|c}
    8 & 2 & 0 &0\\
    2 & 5 & 0 & 0\\
    0 & 0 & 0 & 0
    \end{array}\right]\\
    R'_2 \leftarrow R_2+0.25R_1
    &\left[
    \begin{array}{ccc|c}
    8 & 2 & 0 &0\\
    0 & \frac{9}{2} & 0 & 0\\
    0 & 0 & 0 & 0
    \end{array}\right]
    \end{align}

        hence
        \begin{align}
    8v_1 + 2v_2 &= 0\\
    \frac{9}{2}v_2 &= 0
    \end{align}

        In this case, $v_2 = 0$, hence $v_1=0$, and $v_3$ can take any value. Let $v_3 = 1$, ${\bf v}_3 = (0, 0, 1)^T$, and is already a unit vector

        \begin{align}\hat{\bf v}_3 = \frac{\bf v_3}{\bf |v_3|}=
        \begin{bmatrix} 0 \\ 0 \\ 1\end{bmatrix}\end{align}

4. Form Singular values of ${\bf A}$ from Eigenvalues of ${\bf A}^T\bf A$.

    The Singular values are the squareroot of the eigenvalues, ranked by magnitude along the diagonal such that

    \begin{align}
    {\bf W} = \begin{bmatrix}
    3 & 0 & 0\\
    0 & 2 & 0\\
    0 & 0 & 0 \end{bmatrix}
    \end{align}


5. For Singular vectors $\bf V$ from *eigenvectors*.
 the columns of correspond to the unit eigenvectors in the same sequence as their corresponding eigenvalues,

    i.e. \begin{align}
    {\bf V} &= \begin{bmatrix}{\bf\hat{v}}_1 & {\bf\hat{v}}_2 & {\bf\hat{v}}_3\end{bmatrix}\\
    {\bf V} &= \begin{bmatrix}
    \frac{2}{\sqrt{5}} & \frac{1}{\sqrt{5}} & 0\\
    \frac{1}{\sqrt{5}} & -\frac{2}{\sqrt{5}} & 0\\
    0 & 0 & 1
    \end{bmatrix}
    \end{align}

6. Compute the values of $\bf U$ as
$${\bf u}_i = \frac{1}{\sigma_i}{\bf A}{\bf \hat{v}}_i$$
  such that
  \begin{align}{\bf U}
  &= \begin{bmatrix}{\bf u}_1 & {\bf u}_2 & {\bf u}_3\end{bmatrix}\\
  \end{align}

    * compute ${\bf u}_1$, where $\sigma_1 = \sqrt\lambda_1 = \sqrt{9} = 3$

    \begin{align}
  {\bf u}_1 &= \frac{1}{3}\begin{bmatrix}
    2 & 0 & 0\\
    2 & 1 & 0\\
    0 & -2 & 0
    \end{bmatrix}\cdot\frac{1}{\sqrt{5}}\begin{bmatrix}
    2\\
    1\\
    0
    \end{bmatrix}\\
    {\bf u}_1 &= \frac{1}{3\sqrt{5}}\begin{bmatrix}
    4 \\
    5 \\
    -2
    \end{bmatrix}
    \end{align}

    * compute ${\bf u}_2$ where $\sigma_2 = \sqrt\lambda_2 = \sqrt{4} = 2$

    \begin{align}
  {\bf u}_2 &= \frac{1}{2}\begin{bmatrix}
    2 & 0 & 0\\
    2 & 1 & 0\\
    0 & -2 & 0
    \end{bmatrix}\cdot\frac{1}{\sqrt{5}}\begin{bmatrix}
    1\\
    -2\\
    0
    \end{bmatrix}\\
    {\bf u}_2 &= \frac{1}{2\sqrt{5}}\begin{bmatrix}
    2 \\
    0 \\
    4
    \end{bmatrix}\\
    {\bf u}_2 &= \frac{1}{\sqrt{5}}\begin{bmatrix}
    1 \\
    0 \\
    2
    \end{bmatrix}\\
    \end{align}

    * compute ${\bf u}_3$ where $\sigma_3 = \sqrt\lambda_3 = \sqrt{0} = 0$

    \begin{align}
  {\bf u}_3 &= \frac{1}{0}\begin{bmatrix}
    2 & 0 & 0\\
    2 & 1 & 0\\
    0 & -2 & 0
    \end{bmatrix}\cdot\begin{bmatrix}
    0\\
    0\\
    1
    \end{bmatrix}
    \end{align}

    >### ⚠️ Point of note
    > As $\sigma_3 = 0$, ${\bf u}_3$ can be any vector, as long as it is orthogonal to ${\bf u}_1$ and ${\bf u}_2$.

    The cross-product of any two vectors yields a vector that is orthogonal to the both, hence

    $$\mathbf{u}_3 = \mathbf{u}_1 \times \mathbf{u}_2$$

    which we can calculate as

    \begin{eqnarray}\mathbf{u}_1 \times \mathbf{u}_2 &=& \frac{1}{3\sqrt{5}}\frac{1}{\sqrt{5}} \begin{vmatrix} \bf i & \bf j & \bf k\\
    4 & 5 & -2\\
    1 & 0 & 2
    \end{vmatrix}\\
    &=&  \frac{1}{15} \begin{bmatrix} 10\\
    -10\\
    -5
    \end{bmatrix}\\
    \mathbf{u}_3 &=&  \frac{1}{3} \begin{bmatrix} 2\\
    -2\\
    -1
    \end{bmatrix}
    \end{eqnarray}

    Hence,

    \begin{align}{\bf U}
    &= \begin{bmatrix}
    \frac{4}{3\sqrt{5}} & \frac{1}{\sqrt{5}} & \frac{2}{3}\\
    \frac{5}{3\sqrt{5}} & 0 & \frac{-2}{3}\\
    \frac{-2}{3\sqrt{5}} & \frac{2}{\sqrt{5}} & \frac{-1}{3}\\
    \end{bmatrix}\\
    \end{align}

7. The singular values are $\sigma_k = \sqrt\lambda_i$,

    \begin{align}{\bf \Sigma}
    &= \begin{bmatrix}
    3 & 0 & 0\\
    0 & 2 & 0\\
    0 & 0 & 0\\
    \end{bmatrix}\\
    \end{align}

    >### ⚠️ Point of note
    > The $\sigma_3=0$ tells us that this is a singular system, and that there is no uniquew solution.  

There is a bit of math here, but the SVD and other methods presented here can all be handled numerically. There are numerous implemented function for these. You need to know that they exists, and where and why they might be applicable to your problem.

**To Do**:
Why don't you see if you can use AI to generate a code to calculate the SVD of a matrix and use this example as a test function.